In [ ]:
import json
import re # Bổ sung thư viện Regular Expression để xử lý ký tự rác

# 1. Định nghĩa mapping gộp nhóm (Đã bổ sung toàn bộ tag còn thiếu)
manual_groups = {
    # NHÓM KHẨN CẤP
    "crisis_emergence": ["suicide", "death", "faq_1337085"],
    
    # NHÓM GIAO TIẾP & ĐIỀU HƯỚNG
    "smalltalk_welcome": ["greeting", "morning", "afternoon", "evening"],
    "smalltalk_goodbye": ["night", "goodbye", "thanks", "pandora-useful", "user-meditation", "done"],
    "bot_info": ["about", "skill", "creation", "name", "location"],
    "user_frustration": ["hate-you", "hate-me", "wrong", "stupid", "understand"],
    
    # Đã bổ sung 'learn-mental-health' vào nhóm nhờ tư vấn
    "ask_for_help": ["help", "user-advice", "ask", "learn-mental-health"], 
    "smalltalk_jokes": ["jokes"],
    
    # Đã bổ sung 'no-response' để bot xử lý khi user gửi tin nhắn rỗng
    "bot_fallback": ["no-response"], 
    
    # NHÓM HÀNH ĐỘNG 
    "action_agree": ["user-agree", "learn-more", "meditation"],
    "action_deny": ["no-approach", "not-talking"],
    
    # Đã bổ sung 'casual', 'default', 'neutral-response' vào chung nhóm trò chuyện phiếm/chung chung
    "smalltalk_casual": ["casual", "default", "neutral-response"], 
    "control_switch_topic": ["something-else"],
    "control_repeat": ["repeat"],
    
    # NHÓM CẢM XÚC THỂ HIỆN
    "emo_depression": ["sad", "worthless", "depressed"],
    "emo_anxiety": ["anxious", "stressed", "scared", "problem"],
    "emo_positive": ["happy"],
    "physical_symptom": ["sleep"],
    "social_symptom": ["friends"]
    
    # Lưu ý: 'mental-health-fact' sẽ được tự động xử lý như một FAQ độc lập ở phần dưới
}

# Đảo ngược dictionary để map từ old_tag -> new_tag
reverse_mapping = {}
for new_tag, old_tags in manual_groups.items():
    for old_tag in old_tags:
        reverse_mapping[old_tag] = new_tag

# 2. Đọc file dữ liệu gốc
with open('merged_intents.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

new_intents_dict = {}
faq_deduplication_dict = {}

# 3. Xử lý phân loại và gom nhóm
for intent in dataset['intents']:
    original_tag = intent['tag']
    patterns = intent['patterns']
    responses = intent['responses']
    
    # TRƯỜNG HỢP A: Các tag Cảm xúc & Giao tiếp đã map tay
    if original_tag in reverse_mapping:
        target_tag = reverse_mapping[original_tag]
        if target_tag not in new_intents_dict:
            new_intents_dict[target_tag] = {"tag": target_tag, "patterns": [], "responses": []}
            
        new_intents_dict[target_tag]['patterns'].extend(patterns)
        new_intents_dict[target_tag]['responses'].extend(responses)
        
    # TRƯỜNG HỢP B: Các tag Kiến thức (Fact & FAQ)
    else:
        # BƯỚC LÀM SẠCH TUYỆT ĐỐI CHỐNG LỖI ENCODING:
        # 1. Ghép các response và chuyển về in thường
        raw_text = "".join(sorted(responses)).lower()
        
        # 2. Dùng Regex xóa TẤT CẢ các ký tự không phải là chữ cái hoặc số (bao gồm cả dấu câu, khoảng trắng, ký tự mã hóa lỗi như â€™, Ã¢â‚¬...)
        # Chỉ giữ lại các ký tự a-z, 0-9 để so sánh lõi nội dung
        response_key = re.sub(r'[\W_]+', '', raw_text)
        
        if response_key not in faq_deduplication_dict:
            # Lưu lại tag gốc cho lần gặp đầu tiên
            faq_deduplication_dict[response_key] = {
                "tag": original_tag,
                "patterns": list(patterns),
                "responses": list(responses)
            }
        else:
            # Phát hiện trùng lặp tuyệt đối -> Gộp patterns
            faq_deduplication_dict[response_key]['patterns'].extend(patterns)

# 4. Dọn dẹp câu trùng lặp (duplicate patterns/responses) bên trong từng tag
for data in new_intents_dict.values():
    data['patterns'] = list(set(data['patterns']))
    data['responses'] = list(set(data['responses']))

for data in faq_deduplication_dict.values():
    data['patterns'] = list(set(data['patterns']))

# 5. Xuất file
final_dataset_list = list(new_intents_dict.values()) + list(faq_deduplication_dict.values())
final_dataset = {"intents": final_dataset_list}

with open('final_grouped_intents_v3.json', 'w', encoding='utf-8') as f:
    json.dump(final_dataset, f, ensure_ascii=False, indent=4)

print(f"Tổng số tag ban đầu: {len(dataset['intents'])}")
print(f"Tổng số tag sau khi bọc lót chống lỗi & gom nhóm: {len(final_dataset['intents'])}")

Tổng số tag ban đầu: 178
Tổng số tag sau khi bọc lót chống lỗi & gom nhóm: 129


In [ ]:
import json

# Đọc file bị lỗi trùng lặp và encoding của bạn
with open('final_grouped_intents_v3.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

intents = dataset['intents']

# ==========================================
# 1. FIX LỖI ENCODING (KÝ TỰ RÁC)
# ==========================================
def fix_text(text):
    # Thay thế các ký tự lỗi mã hóa UTF-8 phổ biến thành ký tự chuẩn
    return text.replace('â€™', "'").replace('â€œ', '"').replace('â€', '"').replace('Ã©', 'é')

for intent in intents:
    intent['patterns'] = [fix_text(p) for p in intent['patterns']]
    intent['responses'] = [fix_text(r) for r in intent['responses']]

# ==========================================
# 2. FIX LỖI TRÙNG LẶP & ĐỔI TÊN TAG CHO DỄ NHÌN (SEMANTIC NAMING)
# ==========================================
# Cấu trúc merge_rules mới cực kỳ rõ ràng của bạn
merge_rules = {
    "what_is_mental_health": ["fact-1", "faq_7728364"],
    "what_is_mental_illness": ["faq_1590140", "fact-8"],
    "who_it_affects": ["fact-9", "faq_2110618"],
    "how_to_help_others": ["faq_1619387", "fact-13"],
    "treatment_info": ["faq_1155199", "fact-16"],
    "where_to_get_help": ["faq_4197817", "fact-19"],
    "social_loneliness": ["fact-30", "faq_9100298"]
}

# Biến danh sách intents thành từ điển (Dictionary) để dễ thao tác
intents_dict = {intent['tag']: intent for intent in intents}

# Lặp qua từng nhóm trong luật gộp của bạn
for new_tag_name, old_tags_list in merge_rules.items():
    
    # Bước 2.1: Tạo một tag mới tinh (nếu chưa có) để chứa dữ liệu
    if new_tag_name not in intents_dict:
        intents_dict[new_tag_name] = {
            "tag": new_tag_name, 
            "patterns": [], 
            "responses": []
        }
        
    # Bước 2.2: Đi gom đồ đạc từ các tag cũ
    for old_tag in old_tags_list:
        if old_tag in intents_dict:
            # Bê toàn bộ câu hỏi và câu trả lời của tag cũ sang tag mới
            intents_dict[new_tag_name]['patterns'].extend(intents_dict[old_tag]['patterns'])
            intents_dict[new_tag_name]['responses'].extend(intents_dict[old_tag]['responses'])
            
            # Đã chuyển nhà xong thì phải đập bỏ nhà cũ (xóa sổ tag cũ)
            del intents_dict[old_tag]

# Xử lý trường hợp ngoại lệ: "not really" đang bị lạc trôi ở 2 tag
if 'smalltalk_casual' in intents_dict and 'not really' in intents_dict['smalltalk_casual']['patterns']:
    intents_dict['smalltalk_casual']['patterns'].remove('not really')

# Dọn dẹp trùng lặp cục bộ (xóa các câu giống hệt nhau bên trong 1 tag)
for tag, data in intents_dict.items():
    data['patterns'] = list(set(data['patterns']))
    data['responses'] = list(set(data['responses']))

# ==========================================
# 3. XUẤT FILE HOÀN CHỈNH
# ==========================================
final_data = {"intents": list(intents_dict.values())}
with open('cleaned_intents.json', 'w', encoding='utf-8') as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print(f"Đã xử lý xong! Tổng số tag hiện tại đã giảm xuống còn: {len(final_data['intents'])}")

Đã xử lý xong! Tổng số tag hiện tại đã giảm xuống còn: 122
Dữ liệu đã sạch 100%, sẵn sàng để Generate thêm 30 patterns bằng ChatGPT/Gemini!
